# Dynamic Arrays

A comprehensive guide to dynamic arrays: how they work, why we need them, and their performance characteristics.

**Topics Covered:**
1. Static vs Dynamic Arrays
2. Dynamic Array Implementation
3. Amortized Analysis
4. Python's `list` Under the Hood

---

[← Previous: Stacks and Queues](../../Tier_4_Algorithms_and_Data_Structures/04_Linear_Data_Structures/02_stacks_queues.ipynb) | [Next: Binary Search Trees (BST) →](../../Tier_4_Algorithms_and_Data_Structures/05_Tree_Structures/01_binary_search_trees.ipynb)

---

---

## 1. Static vs Dynamic Arrays

### Why Can't We Just Use Fixed-Size Arrays?

**Static arrays** have a fixed size determined at creation time. This leads to two problems:

1. **Wasted Memory**: If we allocate too much space, we waste memory
2. **Overflow**: If we allocate too little, we can't store all our data

```
Static Array Problem:

Case 1: Over-allocation (waste)
┌───┬───┬───┬───┬───┬───┬───┬───┐
│ A │ B │ C │   │   │   │   │   │  ← 5 slots wasted!
└───┴───┴───┴───┴───┴───┴───┴───┘

Case 2: Under-allocation (overflow)
┌───┬───┬───┬───┐
│ A │ B │ C │ D │  ← FULL! Can't add 'E'!
└───┴───┴───┴───┘
```

**Dynamic arrays** solve this by automatically resizing when needed.

### Real-World Analogy

Think of it like a parking lot:
- **Static**: Fixed number of spaces. When full, cars must go elsewhere.
- **Dynamic**: When lot fills up, you automatically build a bigger lot and move all cars there.

---

## 2. How Does Automatic Resizing Work?

When a dynamic array runs out of space:

1. **Allocate** a new, larger array
2. **Copy** all existing elements to the new array
3. **Free** the old array (in languages with manual memory management)
4. **Insert** the new element

```
Array Resizing Visualization:

Initial: capacity=4, size=4
┌───┬───┬───┬───┐
│ 1 │ 2 │ 3 │ 4 │  ← FULL!
└───┴───┴───┴───┘

append(5) triggers resize:

Step 1: Allocate new array (2x capacity)
┌───┬───┬───┬───┬───┬───┬───┬───┐
│   │   │   │   │   │   │   │   │  capacity=8
└───┴───┴───┴───┴───┴───┴───┴───┘

Step 2: Copy elements
┌───┬───┬───┬───┬───┬───┬───┬───┐
│ 1 │ 2 │ 3 │ 4 │   │   │   │   │
└───┴───┴───┴───┴───┴───┴───┴───┘

Step 3: Add new element
┌───┬───┬───┬───┬───┬───┬───┬───┐
│ 1 │ 2 │ 3 │ 4 │ 5 │   │   │   │  size=5
└───┴───┴───┴───┴───┴───┴───┴───┘
```

### Growth Factor Trade-offs

| Growth Factor | Pros | Cons |
|--------------|------|------|
| **2x** (doubling) | Fewer resizes, simpler math | More wasted space (up to 50%) |
| **1.5x** | Less wasted space (~33%) | More frequent resizes |
| **1.25x** | Even less waste | Many more resizes |

**Common choices:**
- Python `list`: ~1.125x (complex formula, very space-efficient)
- Java `ArrayList`: 1.5x
- C++ `vector`: 2x (GCC) or 1.5x (MSVC)

---

## 3. Dynamic Array Implementation

### Original Implementation (from Algorithms_HW7)

The basic `DynamicMassive` class uses numpy arrays as the underlying storage:

In [ ]:
import numpy as np

# Original DynamicMassive from Algorithms_HW7
class DynamicMassive:
    """Basic dynamic array using numpy for storage."""
    
    def __init__(self):
        self.count = 0
        self.size = 1
        self.A = self._make_array(self.size)
    
    def __len__(self):
        return self.count
    
    def __getitem__(self, k):
        if k < 0 or k >= self.count:
            raise IndexError('Index out of bounds!')
        return self.A[k]
    
    def append(self, elem):
        if self.count == self.size:
            # Double the capacity
            B = self._make_array(2 * self.size)
            for i in range(self.count):
                B[i] = self.A[i]
            self.A = B
            self.size = 2 * self.size
        
        self.A[self.count] = elem
        self.count += 1
    
    def _make_array(self, new_cap):
        return np.empty(new_cap, dtype=int, order='C')

### Enhanced Implementation with Full Features

Here's a complete implementation with all standard operations:

In [ ]:
from typing import Any, Iterator, Optional
import ctypes


class DynamicArray:
    """
    A dynamic array implementation that automatically resizes.
    
    Resize Strategy:
    - Grow: When full, double the capacity (2x growth factor)
    - Shrink: When 1/4 full, halve the capacity (to avoid thrashing)
    
    The 2x growth factor ensures O(1) amortized append operations.
    The 1/4 shrink threshold prevents resize thrashing at boundaries.
    
    Attributes:
        _size: Number of elements currently stored
        _capacity: Total slots available in underlying array
        _array: Raw ctypes array for storage
    """
    
    GROWTH_FACTOR: float = 2.0
    SHRINK_THRESHOLD: float = 0.25  # Shrink when size <= capacity * threshold
    MIN_CAPACITY: int = 1
    
    def __init__(self, initial_capacity: int = 1) -> None:
        """
        Initialize an empty dynamic array.
        
        Args:
            initial_capacity: Starting capacity (default: 1)
        """
        self._size: int = 0
        self._capacity: int = max(initial_capacity, self.MIN_CAPACITY)
        self._array: ctypes.Array = self._make_array(self._capacity)
    
    def __len__(self) -> int:
        """Return the number of elements in the array."""
        return self._size
    
    def __getitem__(self, index: int) -> Any:
        """
        Get element at index. Supports negative indexing.
        
        Args:
            index: Position to access (-size <= index < size)
            
        Returns:
            Element at the specified index
            
        Raises:
            IndexError: If index is out of bounds
        """
        if index < 0:
            index += self._size
        if not 0 <= index < self._size:
            raise IndexError(f'Index {index} out of bounds for size {self._size}')
        return self._array[index]
    
    def __setitem__(self, index: int, value: Any) -> None:
        """
        Set element at index. Supports negative indexing.
        
        Args:
            index: Position to modify (-size <= index < size)
            value: New value to store
            
        Raises:
            IndexError: If index is out of bounds
        """
        if index < 0:
            index += self._size
        if not 0 <= index < self._size:
            raise IndexError(f'Index {index} out of bounds for size {self._size}')
        self._array[index] = value
    
    def __iter__(self) -> Iterator[Any]:
        """Iterate over elements in the array."""
        for i in range(self._size):
            yield self._array[i]
    
    def __repr__(self) -> str:
        """String representation showing contents."""
        elements = [str(self._array[i]) for i in range(self._size)]
        return f"DynamicArray([{', '.join(elements)}])"
    
    @property
    def capacity(self) -> int:
        """Current capacity of the underlying array."""
        return self._capacity
    
    def append(self, value: Any) -> None:
        """
        Add element to the end of the array.
        
        If the array is full, it will be resized to double its capacity.
        This operation is O(1) amortized, O(n) worst case.
        
        Args:
            value: Element to append
        """
        if self._size == self._capacity:
            self._resize(int(self._capacity * self.GROWTH_FACTOR))
        
        self._array[self._size] = value
        self._size += 1
    
    def insert(self, index: int, value: Any) -> None:
        """
        Insert element at specified index.
        
        Elements at and after the index are shifted right.
        This operation is O(n) due to shifting.
        
        Args:
            index: Position to insert at (0 <= index <= size)
            value: Element to insert
            
        Raises:
            IndexError: If index is out of bounds
        """
        if index < 0:
            index += self._size
        if not 0 <= index <= self._size:
            raise IndexError(f'Insert index {index} out of bounds')
        
        if self._size == self._capacity:
            self._resize(int(self._capacity * self.GROWTH_FACTOR))
        
        # Shift elements right
        for i in range(self._size, index, -1):
            self._array[i] = self._array[i - 1]
        
        self._array[index] = value
        self._size += 1
    
    def delete(self, index: int) -> Any:
        """
        Remove and return element at specified index.
        
        Elements after the index are shifted left.
        If size drops below 1/4 capacity, array is shrunk.
        This operation is O(n) due to shifting.
        
        Args:
            index: Position to delete (-size <= index < size)
            
        Returns:
            The removed element
            
        Raises:
            IndexError: If index is out of bounds or array is empty
        """
        if self._size == 0:
            raise IndexError('Cannot delete from empty array')
        
        if index < 0:
            index += self._size
        if not 0 <= index < self._size:
            raise IndexError(f'Delete index {index} out of bounds')
        
        value = self._array[index]
        
        # Shift elements left
        for i in range(index, self._size - 1):
            self._array[i] = self._array[i + 1]
        
        self._size -= 1
        
        # Shrink if too empty (but maintain minimum capacity)
        if (self._size <= self._capacity * self.SHRINK_THRESHOLD 
            and self._capacity > self.MIN_CAPACITY):
            self._resize(max(self._capacity // 2, self.MIN_CAPACITY))
        
        return value
    
    def pop(self) -> Any:
        """
        Remove and return the last element.
        
        Returns:
            The removed element
            
        Raises:
            IndexError: If array is empty
        """
        return self.delete(self._size - 1)
    
    def search(self, value: Any) -> int:
        """
        Find the index of the first occurrence of value.
        
        Args:
            value: Element to search for
            
        Returns:
            Index of the element, or -1 if not found
        """
        for i in range(self._size):
            if self._array[i] == value:
                return i
        return -1
    
    def _resize(self, new_capacity: int) -> None:
        """
        Resize the underlying array to new_capacity.
        
        This is a private method that:
        1. Creates a new array with the specified capacity
        2. Copies all existing elements
        3. Replaces the old array
        
        Time Complexity: O(n) where n is current size
        
        Args:
            new_capacity: The new capacity (must be >= size)
        """
        new_array = self._make_array(new_capacity)
        
        for i in range(self._size):
            new_array[i] = self._array[i]
        
        self._array = new_array
        self._capacity = new_capacity
    
    def _make_array(self, capacity: int) -> ctypes.Array:
        """
        Create a new raw array of Python objects.
        
        Uses ctypes to create a low-level array that can hold
        references to any Python object.
        
        Args:
            capacity: Size of the array to create
            
        Returns:
            A ctypes array of py_object
        """
        return (capacity * ctypes.py_object)()

---

## 4. Complexity Analysis

### Time & Space Complexity Table

| Operation | Worst Case | Amortized | Space |
|-----------|------------|-----------|-------|
| Access `[i]` | O(1) | O(1) | - |
| Append | O(n) | **O(1)** | O(n) |
| Insert at i | O(n) | O(n) | - |
| Delete at i | O(n) | O(n) | - |
| Search | O(n) | O(n) | - |
| Pop (last) | O(n)* | **O(1)** | - |

*Pop is O(n) worst case only when shrinking occurs

---

## 5. Amortized Analysis: Why Append is O(1)

### The Accounting Method

Let's trace what happens when we append 9 elements:

```
Amortized Analysis of append() with 2x growth factor:

Op  Size  Capacity  Cost  Cumulative  Notes
─────────────────────────────────────────────────────────
1   1     1         1     1           
2   2     2         2     3           resize: copy 1 + insert 1
3   3     4         2     5           resize: copy 2 + insert 1
4   4     4         1     6           
5   5     8         5     11          resize: copy 4 + insert 1
6   6     8         1     12          
7   7     8         1     13          
8   8     8         1     14          
9   9     16        9     23          resize: copy 8 + insert 1

Total cost: 23 for 9 operations
Amortized: 23/9 ≈ 2.56 = O(1) per operation!
```

### Why Does This Work?

The key insight: **We resize infrequently enough that the expensive copies are "paid for" by the cheap inserts.**

```
Cost breakdown for n appends:

Resize copies: 1 + 2 + 4 + 8 + ... + n/2 = n - 1 copies
Regular inserts: n inserts (cost 1 each)

Total cost ≤ n + (n - 1) = 2n - 1
Amortized cost = (2n - 1) / n ≈ 2 = O(1)
```

### Visual Demonstration of Amortized Cost

In [ ]:
def demonstrate_amortized_analysis(n: int = 17) -> None:
    """
    Demonstrate amortized analysis by tracking costs.
    
    Args:
        n: Number of append operations to perform
    """
    print(f"Amortized Analysis: {n} append operations\n")
    print(f"{'Op':>3}  {'Size':>4}  {'Cap':>4}  {'Cost':>4}  {'Total':>5}  Notes")
    print("─" * 55)
    
    size = 0
    capacity = 1
    total_cost = 0
    
    for i in range(1, n + 1):
        if size == capacity:
            # Resize needed: cost = copy old elements + insert new
            cost = size + 1
            capacity *= 2
            note = f"resize: copy {size} + insert 1"
        else:
            cost = 1
            note = ""
        
        size += 1
        total_cost += cost
        
        print(f"{i:>3}  {size:>4}  {capacity:>4}  {cost:>4}  {total_cost:>5}  {note}")
    
    print("─" * 55)
    print(f"\nTotal cost: {total_cost} for {n} operations")
    print(f"Amortized cost: {total_cost/n:.2f} per operation = O(1)")


demonstrate_amortized_analysis(17)

---

## 6. Examples: Observing Capacity Changes

In [ ]:
# Step-by-step capacity tracking
arr = DynamicArray()

print("Watching capacity changes during appends:\n")
print(f"{'Operation':<15} {'Size':<6} {'Capacity':<10} {'Array'}")
print("─" * 50)

for i in range(1, 18):
    old_cap = arr.capacity
    arr.append(i)
    
    resize_marker = " ← RESIZE!" if arr.capacity != old_cap else ""
    print(f"append({i:<2})      {len(arr):<6} {arr.capacity:<10} {list(arr)}{resize_marker}")

In [ ]:
# Demonstrating shrink behavior
print("\nWatching capacity changes during deletions:\n")
print(f"{'Operation':<15} {'Size':<6} {'Capacity':<10} {'Array'}")
print("─" * 50)

arr = DynamicArray()
for i in range(1, 17):
    arr.append(i)

print(f"{'Initial':<15} {len(arr):<6} {arr.capacity:<10} {list(arr)}")
print()

while len(arr) > 0:
    old_cap = arr.capacity
    val = arr.pop()
    
    shrink_marker = " ← SHRINK!" if arr.capacity != old_cap else ""
    print(f"pop()→{val:<8} {len(arr):<6} {arr.capacity:<10} {list(arr)}{shrink_marker}")

---

## 7. Performance Comparison

In [ ]:
import time
import numpy as np


def time_appends(n: int = 100000) -> dict:
    """
    Compare append performance of different array types.
    
    Args:
        n: Number of elements to append
        
    Returns:
        Dictionary with timing results
    """
    results = {}
    
    # Python list
    start = time.perf_counter()
    py_list = []
    for i in range(n):
        py_list.append(i)
    results['Python list'] = time.perf_counter() - start
    
    # Our DynamicArray
    start = time.perf_counter()
    dyn_arr = DynamicArray()
    for i in range(n):
        dyn_arr.append(i)
    results['DynamicArray'] = time.perf_counter() - start
    
    # NumPy with resize (simulating dynamic behavior)
    start = time.perf_counter()
    np_arr = np.array([], dtype=int)
    for i in range(n):
        np_arr = np.append(np_arr, i)  # Creates new array each time!
    results['NumPy append (naive)'] = time.perf_counter() - start
    
    # Pre-allocated NumPy (for comparison - this is cheating)
    start = time.perf_counter()
    np_fixed = np.empty(n, dtype=int)
    for i in range(n):
        np_fixed[i] = i
    results['NumPy pre-allocated'] = time.perf_counter() - start
    
    return results


# Run with smaller n to keep it fast
n = 10000
print(f"Timing {n:,} append operations:\n")

results = time_appends(n)

print(f"{'Implementation':<25} {'Time (s)':<12} {'Relative'}")
print("─" * 50)

baseline = results['Python list']
for name, elapsed in sorted(results.items(), key=lambda x: x[1]):
    relative = elapsed / baseline
    print(f"{name:<25} {elapsed:<12.4f} {relative:.1f}x")

### Why is NumPy `append` So Slow?

`np.append()` creates a **new array every time**, copying all elements. This gives O(n²) total time for n appends!

```
np.append behavior (BAD - don't do this!):

append(1): [1]                    copy 0 elements
append(2): [1, 2]                 copy 1 element
append(3): [1, 2, 3]              copy 2 elements
append(4): [1, 2, 3, 4]           copy 3 elements
...
append(n): [1, 2, ..., n]         copy n-1 elements

Total copies: 0 + 1 + 2 + ... + (n-1) = n(n-1)/2 = O(n²)
```

**Lesson**: If you know the size upfront, pre-allocate. If not, use Python `list` and convert to NumPy at the end.

---

## 8. Python's `list` Under the Hood

### CPython's List Implementation

Python's `list` is implemented in C as a dynamic array with a sophisticated over-allocation strategy.

#### Over-allocation Formula (CPython 3.x)

When resizing, CPython uses this formula to determine new capacity:

```c
new_allocated = (size_t)newsize + (newsize >> 3) + (newsize < 9 ? 3 : 6);
```

This translates to approximately:
- **Growth factor**: ~1.125 (12.5% over-allocation)
- **Small lists**: Add 3-6 extra slots
- **Large lists**: Over-allocate by ~12.5%

#### Why ~1.125x Instead of 2x?

1. **Memory efficiency**: Maximum ~12.5% wasted space vs up to 50% with 2x
2. **Cache friendliness**: Smaller arrays fit better in CPU cache
3. **Trade-off**: More frequent resizes, but each resize is smaller

In [ ]:
import sys


def explore_python_list_internals():
    """Explore how Python list allocates memory."""
    
    print("Python list memory allocation pattern:\n")
    print(f"{'Size':<8} {'Bytes':<12} {'Bytes/elem':<12} {'Capacity*'}")
    print("─" * 50)
    
    lst = []
    prev_size = sys.getsizeof(lst)
    
    for i in range(50):
        lst.append(i)
        current_size = sys.getsizeof(lst)
        
        # Only print when allocation changes
        if current_size != prev_size or i < 10:
            bytes_per_elem = (current_size - 56) / len(lst)  # 56 is base list overhead
            # Estimate capacity from size (8 bytes per pointer on 64-bit)
            estimated_cap = (current_size - 56) // 8
            
            resize = " ← resize" if current_size != prev_size and i > 0 else ""
            print(f"{len(lst):<8} {current_size:<12} {bytes_per_elem:<12.1f} ~{estimated_cap}{resize}")
            prev_size = current_size
    
    print("\n* Capacity estimated from memory size (8 bytes per pointer)")
    print("  Base list overhead is ~56 bytes on 64-bit Python")


explore_python_list_internals()

### Key Takeaways About Python Lists

1. **Dynamic arrays**: Python lists ARE dynamic arrays, just like our implementation
2. **Optimized growth**: Uses ~1.125x growth for memory efficiency
3. **O(1) amortized append**: Despite smaller growth factor, still constant amortized time
4. **Pointer storage**: Lists store pointers to objects, not objects themselves
5. **Over-allocation**: Always allocates a bit extra to reduce resize frequency

---

## 9. Summary

### What We Learned

```
Dynamic Array Key Concepts:

┌─────────────────────────────────────────────────────────────┐
│  1. RESIZING STRATEGY                                       │
│     • Grow by constant factor (1.5x-2x) when full           │
│     • Shrink when too empty (typically at 1/4 capacity)     │
│     • Trade-off: memory vs resize frequency                 │
├─────────────────────────────────────────────────────────────┤
│  2. AMORTIZED ANALYSIS                                      │
│     • Append is O(1) amortized, O(n) worst case             │
│     • Expensive resizes "paid for" by cheap inserts         │
│     • Total cost of n appends ≤ 2n                          │
├─────────────────────────────────────────────────────────────┤
│  3. PYTHON'S LIST                                           │
│     • Uses ~1.125x growth factor                            │
│     • Very memory efficient                                 │
│     • Implemented in optimized C                            │
└─────────────────────────────────────────────────────────────┘
```

### When to Use Dynamic Arrays

| Use Case | Recommendation |
|----------|----------------|
| Unknown size, many appends | ✅ Dynamic array |
| Known size, fixed data | ❌ Use static array (NumPy) |
| Many insertions in middle | ❌ Consider linked list |
| Need fast random access | ✅ Dynamic array |

In [ ]:
# Final demonstration: our DynamicArray in action
print("DynamicArray Demo\n")

arr = DynamicArray()

# Append elements
for x in [10, 20, 30, 40, 50]:
    arr.append(x)
print(f"After appends: {arr}")
print(f"Size: {len(arr)}, Capacity: {arr.capacity}\n")

# Insert at position
arr.insert(2, 25)
print(f"After insert(2, 25): {arr}\n")

# Access and modify
print(f"arr[0] = {arr[0]}")
print(f"arr[-1] = {arr[-1]}")
arr[0] = 100
print(f"After arr[0] = 100: {arr}\n")

# Delete
deleted = arr.delete(2)
print(f"Deleted element at index 2: {deleted}")
print(f"After delete: {arr}\n")

# Search
idx = arr.search(40)
print(f"Index of 40: {idx}")
print(f"Index of 999: {arr.search(999)}")

---

[← Previous: Stacks and Queues](../../Tier_4_Algorithms_and_Data_Structures/04_Linear_Data_Structures/02_stacks_queues.ipynb) | [Next: Binary Search Trees (BST) →](../../Tier_4_Algorithms_and_Data_Structures/05_Tree_Structures/01_binary_search_trees.ipynb)